In [2]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
import os

PROJECT_ROOT = Path(os.getcwd()).parent  # data/ -> project root

with open(r"C:\Users\udaya\OneDrive\Desktop\Major Project\New\Adaptive-IDS-with-XAI\models\top_features.pkl", "rb") as f:
    top_features = pickle.load(f)

TEST_FILES = [r"C:\Users\udaya\OneDrive\Desktop\Major Project\New\Adaptive-IDS-with-XAI\archive\02-14-2018.csv",
              r"C:\Users\udaya\OneDrive\Desktop\Major Project\New\Adaptive-IDS-with-XAI\archive\02-15-2018.csv",
              r"C:\Users\udaya\OneDrive\Desktop\Major Project\New\Adaptive-IDS-with-XAI\archive\02-16-2018.csv",
              r"C:\Users\udaya\OneDrive\Desktop\Major Project\New\Adaptive-IDS-with-XAI\archive\02-20-2018.csv",

   r"C:\Users\udaya\OneDrive\Desktop\Major Project\New\Adaptive-IDS-with-XAI\archive\02-28-2018.csv",
    r"C:\Users\udaya\OneDrive\Desktop\Major Project\New\Adaptive-IDS-with-XAI\archive\03-01-2018.csv",
    r"C:\Users\udaya\OneDrive\Desktop\Major Project\New\Adaptive-IDS-with-XAI\archive\03-02-2018.csv",
]

chunks = []
for fp in TEST_FILES:
    df_temp = pd.read_csv(fp, low_memory=False, nrows=20000)
    df_temp.columns = df_temp.columns.str.strip()
    df_temp = df_temp[df_temp["Label"] != "Label"]
    chunks.append(df_temp)

df = pd.concat(chunks, ignore_index=True)

df["Fwd Pkts/s"] = pd.to_numeric(df["Fwd Pkts/s"], errors='coerce')
df["Flow Pkts/s"] = pd.to_numeric(df["Flow Pkts/s"], errors='coerce')
df["Flow IAT Std"] = pd.to_numeric(df["Flow IAT Std"], errors='coerce')
df["Flow IAT Mean"] = pd.to_numeric(df["Flow IAT Mean"], errors='coerce')

# Compute derived features
df["pkt_rate_ratio"] = df["Fwd Pkts/s"] / (df["Flow Pkts/s"] + 1)
df["iat_variation"] = df["Flow IAT Std"] / (df["Flow IAT Mean"] + 1)
df["label"] = df["Label"].apply(
    lambda x: 0 if str(x).strip().lower() == "benign" else 1
)
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=top_features + ["label"])

output_path = r"C:\Users\udaya\OneDrive\Desktop\Major Project\New\Adaptive-IDS-with-XAI\data\retrain_buffer.csv"
df[top_features + ["label"]].to_csv(output_path, index=False)
print("Buffer saved:", df.shape)
print("Saved to:", output_path)

Buffer saved: (139453, 87)
Saved to: C:\Users\udaya\OneDrive\Desktop\Major Project\New\Adaptive-IDS-with-XAI\data\retrain_buffer.csv
